# 113. RG3 미성형 전용 공정 전문가 앙상블
정상566 + 미성형8, 가스17 제외. 외부 관측구간을 전처리·정상참조·모델·피처조합·임계값 선택에서 제외한다. 이미 반복 확인한 자료의 개발 중첩 구간검증이며 새 독립자료의 검증은 아니다.
가이드북 PDF11쪽 및 https://link.springer.com/article/10.1007/s00170-023-11100-1 참고. 캐비티 압력곡선이 없으므로 문헌의 곡선모델을 재현한 것이 아니라 요약 센서 대리피처를 시험한다. 단계별 Git 참고: https://github.com/Chohyungun/NOVA50101-KAMP-Injection-Molding .
5개 별도 피처표현 각6개. RF/ExtraTrees 2seed 평균과 로지스틱의 평균. 학습 내부 F1로 단독·두 전문가 평균15조합과 임계값을 선택한다. 미래 회복·Timestamp·구간ID를 모델피처로 넣지 않는다.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, json
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix, average_precision_score, precision_recall_curve
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3') & (d.PassOrFail.eq(0)|d.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int); g=d.comparison_block.to_numpy(); blocks=np.unique(g)
assert len(d)==574 and y.sum()==8 and len(blocks)==5
assert d.TimeStamp.is_unique
sensors=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Filling_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Speed']
raw=d[sensors].copy()
delta=d[[v+'__delta1' for v in sensors]].copy()
phys=pd.DataFrame({'pressure_gap':d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure,
 'fill_fraction':d.Filling_Time/d.Injection_Time,
 'cushion_meter_fraction':d.Cushion_Position/d.Plasticizing_Position,
 'meter_minus_cushion_change':d.Plasticizing_Position__delta1-d.Cushion_Position__delta1,
 'barrel_mold_gap':d.Barrel_Temperature_5-(d.Mold_Temperature_3+d.Mold_Temperature_4)/2,
 'fill_time_per_speed':d.Filling_Time/d.Max_Injection_Speed})
seqcols=['Max_Injection_Pressure_path_efficiency','Cushion_Position_path_efficiency','Barrel_Temperature_5_path_efficiency','pressure_recent3_energy_share','pressure_cushion_delta_sync','pressure_then_cushion_lag_advantage']
seq=pd.read_csv(t/'rg3_shortshot_sequence_features.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[d.TimeStamp,seqcols].reset_index(drop=True)
# No next-shot recovery, future labels, or observation block ID in model inputs.
views={'raw6':raw,'current_delta6':delta,'phase_proxy6':phys,'sequence6':seq,'conditional_phase6':phys}
names=list(views); memo={}; audits=[]
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 result=[]
 for name,x in views.items():
  a=x.replace([np.inf,-np.inf],np.nan).to_numpy(float)
  normal=ref[y[ref]==0]
  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal])
  a=imp.transform(a)
  if name=='conditional_phase6':
   context=d[['Barrel_Temperature_5','Mold_Temperature_3','Max_Injection_Speed','Hopper_Temperature']].to_numpy()
   ci=SimpleImputer(strategy='median').fit(context[normal]); context=ci.transform(context)
   sc=RobustScaler().fit(context[normal]); context=sc.transform(context)
   nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(context[normal])
   idx=nn.kneighbors(context[np.r_[ref,q]],return_distance=False)
   floor=np.maximum(a[normal].std(axis=0)*.1,1e-6)
   transformed=[]
   for row,j in zip(idx,np.r_[ref,q]):
    peers=normal[row]; peers=peers[peers!=j][:30]
    transformed.append((a[j]-np.median(a[peers],axis=0))/np.maximum(a[peers].std(axis=0),floor))
   tr=np.array(transformed[:len(ref)]);te=np.array(transformed[len(ref):])
  else:tr=a[ref];te=a[q]
  scores=[]
  if len(np.unique(y[ref]))<2:
   result.append(np.zeros(len(q)));continue
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
    scores.append(m.fit(tr,y[ref]).predict_proba(te)[:,1])
  sc=RobustScaler().fit(tr[y[ref]==0])
  lr=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  lin=lr.fit(sc.transform(tr),y[ref]).predict_proba(sc.transform(te))[:,1]
  result.append((np.mean(scores,axis=0)+lin)/2)
 out=np.column_stack(result);memo[key]=out
 audits.append(dict(train=len(ref),train_shortshot=int(y[ref].sum()),test=len(q),timestamp_overlap=0))
 return out
weights=[]
for j,name in enumerate(names):
 w=np.zeros(len(names));w[j]=1;weights.append((name,w))
for a in range(len(names)):
 for b in range(a+1,len(names)):
  w=np.zeros(len(names));w[a]=w[b]=.5;weights.append((names[a]+'+'+names[b],w))
def choose_threshold(yy,score):
 precision,recall,thresholds=precision_recall_curve(yy,score)
 f=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
 k=max(range(len(thresholds)),key=lambda k:(f[k],recall[k],thresholds[k]))
 return float(thresholds[k]),float(f[k])
records=[]; choices=[]
for block in blocks:
 q=np.flatnonzero(g==block);ref=np.flatnonzero(g!=block)
 outer=predict(ref,q);inner=np.full((len(ref),len(names)),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib
  inner[mask]=predict(ref[~mask],ref[mask])
 assert np.isfinite(inner).all()
 candidates=[]
 for name,w in weights:
  cut,fi=choose_threshold(y[ref],inner@w)
  candidates.append((fi,name,w,cut))
  pr=(outer@w)>=cut
  for k,i in enumerate(q):
   records.append(dict(model=name,TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float((outer@w)[k]),threshold=cut,prediction=int(pr[k])))
 best=max(candidates,key=lambda r:(r[0],-np.count_nonzero(r[2]),r[1]))
 fi,name,w,cut=best;score=outer@w;pr=score>=cut
 choices.append(dict(block=block,selected=name,inner_F1=fi,threshold=cut))
 for k,i in enumerate(q):
  records.append(dict(model='primary_inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float(score[k]),threshold=cut,prediction=int(pr[k])))
 print('Completed',block,'selected',name,flush=True)
p=pd.DataFrame(records);rows=[]
for name,z in p.groupby('model'):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
p.to_csv(t/'rg3_shortshot_specialist_predictions.csv',index=False,encoding='utf-8-sig')
m.to_csv(t/'rg3_shortshot_specialist_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(t/'rg3_shortshot_specialist_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audits).to_csv(t/'rg3_shortshot_specialist_training_audit.csv',index=False,encoding='utf-8-sig')
case=d.loc[y==1,['TimeStamp','comparison_block']].merge(p[p.model.eq('primary_inner_selected')][['TimeStamp','prediction','score','threshold']],on='TimeStamp',validate='one_to_one')
case.to_csv(t/'rg3_shortshot_specialist_all8_cases.csv',index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False),flush=True)
print('ALL 8',case.to_string(index=False),flush=True)


Completed 20201021_001626_RG3 selected sequence6+conditional_phase6


Completed 20201022_002405_RG3 selected phase_proxy6


Completed 20201022_044026_RG3 selected raw6+conditional_phase6


Completed 20201023_010128_RG3 selected conditional_phase6


Completed 20201023_044148_RG3 selected sequence6


                            model  TP  FP  FN  TN       F1       AP
  phase_proxy6+conditional_phase6   2  54   6 512 0.062500 0.023588
          raw6+conditional_phase6   3  95   5 471 0.056604 0.017752
         current_delta6+sequence6   4 188   4 378 0.040000 0.018317
current_delta6+conditional_phase6   7 341   1 225 0.039326 0.017203
                   raw6+sequence6   2  99   6 467 0.036697 0.023716
      current_delta6+phase_proxy6   5 264   3 302 0.036101 0.018380
     sequence6+conditional_phase6   2 107   6 459 0.034188 0.018856
                   current_delta6   7 395   1 171 0.034146 0.016084
                     phase_proxy6   2 109   6 457 0.033613 0.026950
                raw6+phase_proxy6   3 237   5 329 0.024194 0.020366
                             raw6   3 252   5 314 0.022814 0.015968
           phase_proxy6+sequence6   1  80   7 486 0.022472 0.025672
              raw6+current_delta6   3 313   5 253 0.018519 0.016494
                        sequence6   2 216   6 35

ALL 8           TimeStamp    comparison_block  prediction    score  threshold
2020-10-22 01:09:23 20201022_002405_RG3           0 0.309468   0.704663
2020-10-22 04:48:39 20201022_044026_RG3           0 0.303245   0.542587
2020-10-22 05:42:13 20201022_044026_RG3           0 0.272995   0.542587
2020-10-22 06:12:03 20201022_044026_RG3           0 0.360910   0.542587
2020-10-22 06:58:22 20201022_044026_RG3           0 0.335404   0.542587
2020-10-23 02:25:54 20201023_010128_RG3           0 0.356786   0.635732
2020-10-23 05:47:40 20201023_044148_RG3           0 0.496049   0.656944
2020-10-23 05:54:52 20201023_044148_RG3           0 0.217858   0.656944
